## Aula 08 - 20/10/2026

Na Aula 07 filtramos um único DataFrame (o espaço amostral Ω) com máscaras booleanas. Sistemas reais, porém, não guardam dados em uma tabela única: eles os distribuem em múltiplas entidades que se relacionam por **chaves** (identificadores únicos). Cruzar essas entidades exige a **Álgebra Relacional** de Codd (1970): união (∪), interseção (∩), diferença (−), produto cartesiano (×), seleção (σ) e projeção (π).

As operações de conjuntos foram estudadas em seções passadas, visto que no momento estudaremos as operações ainda não definidas em nossos materiais. A primeira operação a ser definida serpa a operação de Seleção ou Select.

### Select - σ

A operação de seleção irá, a partir de uma tabela original, criar uma nova tabela que satisfaça uma certa propriedade de algum atributo em particular (linhas).

### Project - π

A operação de projeção irá, a partir de uma tabela original, uma nova tabela composta por determinadas colunas desta.

### Join

O Join refere-se à operações dos conjuntos. Ela é realizada em função de um atributo em comum entre as tabelas. A operação irá formar o produto cartesiano de todas as n-uplas - ou linhas - da primeira relação com todas as k-uplas - ou linhas - da segunda relação, resultando em um conjunto com (n+k)-uplas, selecionando destre estes os que possuem atributo comum com o mesmo valor.

Tópcio a analisar: SQL e demais atribuições em nosso [Livro texto](https://animaedu-my.sharepoint.com/:b:/r/personal/ciro_chaves_ulife_com_br/Documents/Matem%C3%A1tica%20Computacional%20Aplicada%202026-2/Livro%20texto/Fundamentos%20Matem%C3%A1ticos%20para%20Ci%C3%AAncias%20da%20Computa%C3%A7%C3%A3o.pdf?d=w983aa8dbb1e140b5818df8d46dedb708&csf=1&web=1&e=zjv0uQ).

*Lembre-se de que só com o seu e-mail institucional você pode abrir a URL acima.*

### Live coding 1 - [Slide da Aula #8 PRÁTICA UNIFG](https://animaedu-my.sharepoint.com/:p:/r/personal/ciro_chaves_ulife_com_br/Documents/Matem%C3%A1tica%20Computacional%20Aplicada%202026-2/Slides%20-%20Pr%C3%A1tica/Aula%20%238%20PR%C3%81TICA%20UNIFG%20-%20%C3%81lgebra%20Relacional%20Pandas.pptx?d=wf8c3ef7a12ca4799acf6b610f4fc658b&csf=1&web=1&e=2LAcvU)

## O cenário: duas fontes de verdade que divergem

Toda operação de infraestrutura mantém, no mínimo, **dois registros sobre os mesmos ativos**:

- O **inventário** é a planilha de cadastro de cada servidor provisionado, que recebe um `id_srv` e um IP no momento do cadastro.
- O **monitoramento** é a tela do software de monitoramento: Ela diz o que funciona no momento, de acordo com o seu `status`.

Essas tabelas são alimentadas por rotinas diferentes — o provisionamento instala o sistema operacional e cadastra a máquina no inventário; a instalação do agente de monitoramento é outra etapa, de outra equipe, em outro dia.

E nenhuma das duas tabelas, sozinha, responde às perguntas que importam:

1. **Cobertura** — quais servidores cadastrados estão sem monitoramento? (pontos cegos da operação);
2. **Órfãos** — quais máquinas reportam métricas sem constar no inventário? (shadow IT);
3. **Divergência total** — quantos ativos estão fora da interseção, somando os dois lados?

Só o cruzamento relacional responde. Nos próximos blocos, adotamos a convenção: **A = inventário (`servidores`)** e **B = monitoramento (`monitorados`)**.

In [26]:
import pandas as pd

# Conjunto A: Inventário de Servidores
servidores = pd.DataFrame(
    {
        "id_srv": ["SRV_01", "SRV_02", "SRV_03", "SRV_04"],
        "ip": ["10.0.0.1", "10.0.0.2", "10.0.0.3", "10.0.0.4"],
    }
)

# Conjunto B: Monitoramento Ativo
monitorados = pd.DataFrame(
    {
        "id_srv": ["SRV_02", "SRV_03", "SRV_05"],
        "status": ["ONLINE", "ONLINE", "ALERTA"],
    }
)


In [27]:
servidores.head()

,id_srv,ip
0,SRV_01,10.0.0.1
1,SRV_02,10.0.0.2
2,SRV_03,10.0.0.3
3,SRV_04,10.0.0.4


In [28]:
monitorados.head()

,id_srv,status
0,SRV_02,ONLINE
1,SRV_03,ONLINE
2,SRV_05,ALERTA


Note a assimetria deliberada entre os conjuntos: `SRV_01` e `SRV_04` existem no inventário (A) mas **não** estão no monitoramento (B); `SRV_05` está em B mas **não** consta em A. Essa configuração permite observar o comportamento de cada tipo de join:

- **Inner (A ∩ B):** Retorna apenas as chaves presentes nas duas tabelas → `SRV_02`, `SRV_03`.
- **Outer (A ∪ B):** Retorna todas as chaves; as ausências viram `NaN`.
- **Left (domínio de A):** mantém 100% da tabela A e agrega dados de B quando existirem.
- **Cross (A × B):** todos os pares ordenados possíveis, |A| · |B| linhas.


In [29]:
df_inner = pd.merge(
    servidores,
    monitorados,
    on="id_srv",
    how="inner",
)
print(df_inner)

print('\n\n')

df_outer = pd.merge(
    servidores,
    monitorados,
    on="id_srv",
    how="outer",
)
print(df_outer)


   id_srv        ip  status
0  SRV_02  10.0.0.2  ONLINE
1  SRV_03  10.0.0.3  ONLINE



   id_srv        ip  status
0  SRV_01  10.0.0.1     NaN
1  SRV_02  10.0.0.2  ONLINE
2  SRV_03  10.0.0.3  ONLINE
3  SRV_04  10.0.0.4     NaN
4  SRV_05       NaN  ALERTA


**Problema de auditoria:** quais servidores estão cadastrados no inventário (A) mas **não** possuem monitoramento ativo (B)?

Matematicamente, é a diferença **A − B**, conhecida em bancos de dados como **Anti-Join**. O Pandas não tem um método `anti_join`, então a solução vetorizada combina:

1. `.isin()` — produz um vetor booleano de pertinência (cada chave de A está em B?);
2. `~` — operador lógico de negação (NOT), que inverte a máscara e filtra a diferença.


In [30]:
mascara = servidores["id_srv"].isin(monitorados["id_srv"])
print(mascara.tolist())
print('\n')

sem_agente = servidores[~mascara]
print(sem_agente)


[False, True, True, False]


   id_srv        ip
0  SRV_01  10.0.0.1
3  SRV_04  10.0.0.4


O parâmetro `indicator=True` adiciona ao resultado a coluna especial `_merge`, que classifica matematicamente a origem de cada linha em uma única operação:

- `both` — pertence à interseção (A ∩ B);
- `left_only` — exclusivo da esquerda (A − B);
- `right_only` — exclusivo da direita (B − A).

É a ferramenta padrão para auditorias de consistência entre sistemas: em vez de três queries, um único Outer Merge rotulado.

In [31]:
auditoria = pd.merge(
    servidores,
    monitorados,
    on="id_srv",
    how="outer",
    indicator=True,
)

print(auditoria)

   id_srv        ip  status      _merge
0  SRV_01  10.0.0.1     NaN   left_only
1  SRV_02  10.0.0.2  ONLINE        both
2  SRV_03  10.0.0.3  ONLINE        both
3  SRV_04  10.0.0.4     NaN   left_only
4  SRV_05       NaN  ALERTA  right_only


*Observação importante:*

Conjuntos matemáticos não admitem repetição; DataFrames, sim. Antes de operar com cardinalidades de chaves, reduza a tabela (ou coluna) a um conjunto próprio:

```python
df_unico = df.drop_duplicates()
```

Esse detalhe será decisivo na Missão 03 do desafio: sem `drop_duplicates()`, contagens de chaves ficam infladas por eventos repetidos.